# 06 — Minigame Probability Estimation from Stat Counts

Uses cumulative stat-icon counts from a tracked video to estimate the probability
of each end-of-round minigame, given that item drop rates depend on which minigame
is active.

## Background

In Kirby Air Ride, the minigame played at the end of each round determines which
stat upgrades appear during the round. Observing which stat icons appear on screen
provides evidence about which minigame is active — this is modelled as a Bayesian
update.

## Workflow

1. Define per-minigame item drop likelihoods `P(stat | minigame)`
2. Define a uniform prior over minigames
3. Given observed stat counts, compute `P(minigame | counts)` via Bayes' rule
4. Replay the posterior frame-by-frame and compare tracker vs ground truth

## 0 — Configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_ROOT = Path("/home1/hendersonj6179@cgu.edu")

with open(REPO_ROOT / "data/splits/splits.json") as f:
    splits = json.load(f)
CLASSES = splits["classes"]   # e.g. ["Boost","Charge","Defense",...]
N_CLASSES = len(CLASSES)

print(f"Stat classes ({N_CLASSES}): {CLASSES}")

## 1 — Minigame Definitions & Likelihood Model

Each minigame is associated with a different item drop distribution.
`LIKELIHOODS[minigame][stat]` = P(a random drop is `stat` | `minigame` is active).

> **TODO:** Replace the placeholder distributions with the real values once confirmed.

In [ ]:
# ── Minigame names ─────────────────────────────────────────────────────────
# TODO: Replace with the actual minigame names
MINIGAMES = [
    "Single Race",
    "Drag Race",
    "Destruction Derby",
]

# ── Prior over minigames (uniform until you have better data) ──────────────
PRIOR = np.ones(len(MINIGAMES)) / len(MINIGAMES)

# ── Likelihood: P(stat_class | minigame) ──────────────────────────────────
# Shape: (n_minigames, n_stat_classes)
# Each row must sum to 1.
# TODO: Fill in the actual drop-rate distributions per minigame.
LIKELIHOODS = np.array([
    # Boost  Charge  Defense  Glide   HP    Offense  TopSpeed  Turn    Weight
    [0.083,0.083,  0.083,   0.083,  0.075,  0.062,   0.083,    0.083,  0.083],  # Single Race
    [0.082,0.082,  0.082,   0.082,  0.074,  0.082,   0.082,    0.082,  0.074],  # Drag Race
    [0.09, 0.072,  0.108,   0.072,  0.072,  0.129,    0.05,    0.072,  0.072],  # Destruction Derby
])

# Normalise each row so it sums to exactly 1
# (allows entering raw counts or unnormalised weights above)
LIKELIHOODS = LIKELIHOODS / LIKELIHOODS.sum(axis=1, keepdims=True)

# Sanity check
assert LIKELIHOODS.shape == (len(MINIGAMES), N_CLASSES), "Shape mismatch"
assert np.allclose(LIKELIHOODS.sum(axis=1), 1.0), "Rows must sum to 1"
print("Likelihood table:")
df_lik = pd.DataFrame(LIKELIHOODS, index=MINIGAMES, columns=CLASSES)
display(df_lik.style.format("{:.3f}").background_gradient(axis=1, cmap="YlOrRd"))

## 2 — Bayesian Update

Given cumulative observation counts `obs[c]` (times stat `c` was seen):

```
P(minigame | obs) ∝ P(minigame) × ∏_c P(stat=c | minigame)^obs[c]
```

Computed in log-space for numerical stability. The canonical implementation lives
in `helpers.py` and is imported here.

In [ ]:
from helpers import compute_posterior

## 3 — Load Tracking Results & Compute Per-Frame Posterior

Reads a tracking CSV with columns `[frame, class_id]` and replays the posterior
update frame by frame, accumulating stat counts as each new track is first seen.

In [ ]:
# ── Run baseline tracker on a chosen video & accumulate posteriors ────────
# Dispatches to boxmot (REID_OPTION='B') or Ultralytics tracker based on arch config,
# exactly mirroring the logic in notebook 04/05.
import yaml as _yaml06
from ultralytics import YOLO as _YOLO06

# ── Load arch config ──────────────────────────────────────────────────────
_ARCH_CFG_06 = REPO_ROOT / 'configs' / 'kartector_botsort_arch.json'
if _ARCH_CFG_06.exists():
    import json as _j06
    _arch06       = _j06.loads(_ARCH_CFG_06.read_text())
    _weights06    = REPO_ROOT / _arch06.get('weights', 'runs/yolo26/final/weights/best.pt')
    _tracker_yaml06 = REPO_ROOT / _arch06.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml')
    _conf06       = _arch06.get('conf', 0.25)
    _reid_opt06   = _arch06.get('reid_option', 'none')   # 'A', 'B', or 'none'
    _reid_b06     = _arch06.get('reid_b_params', {})
else:
    _weights06    = REPO_ROOT / 'runs/yolo26/final/weights/best.pt'
    _tracker_yaml06 = REPO_ROOT / 'configs/kartector_botsort_reentry.yaml'
    _conf06       = 0.25
    _reid_opt06   = 'B'   # notebook 04 chose boxmot by default
    _reid_b06     = {}

# ── Choose a video to analyse ─────────────────────────────────────────────
CHUNKS_DIR_06 = REPO_ROOT / 'data/processed/labelstudiochunks'
_chunk_videos = sorted(CHUNKS_DIR_06.glob('*.mp4'))
DEMO_VIDEO_06 = _chunk_videos[0] if _chunk_videos else None  # ← change index to pick another

df_tracks06 = None
if DEMO_VIDEO_06 is None or not _weights06.exists():
    print('No video or weights found — skipping live tracking.')
else:
    print(f'Running baseline tracker [{_reid_opt06}] on: {DEMO_VIDEO_06.name}')
    print(f'  weights : {_weights06}')
    print(f'  conf    : {_conf06}')
    _rows06 = []

    if _reid_opt06 == 'B':
        # ── boxmot BotSort path (same as notebook 05 REID_OPTION='B') ─────
        import torch as _torch06
        import cv2 as _cv06
        import numpy as _np06
        from boxmot.trackers.botsort.botsort import BotSort as _BotSort06
        from boxmot.utils import WEIGHTS as _BW06
        _device06 = _torch06.device(
            'cuda' if _torch06.cuda.is_available() else
            'mps'  if _torch06.backends.mps.is_available() else 'cpu')
        _det06 = _YOLO06(str(_weights06))
        _tracker06 = _BotSort06(
            reid_weights=_BW06 / 'osnet_x0_25_msmt17.pt',
            device=_device06, half=False,
            track_buffer=_reid_b06.get('track_buffer', 30),
            match_thresh=_reid_b06.get('match_thresh', 0.70),
            appearance_thresh=_reid_b06.get('appearance_thresh', 0.40),
            proximity_thresh=_reid_b06.get('proximity_thresh', 0.5),
            with_reid=True,
        )
        _cap06 = _cv06.VideoCapture(str(DEMO_VIDEO_06))
        _fi06 = 0
        while True:
            _ret06, _frame06 = _cap06.read()
            if not _ret06: break
            _fi06 += 1
            _res06 = _det06(_frame06, conf=_conf06, verbose=False)
            _dets06 = _res06[0].boxes
            if _dets06 is not None and len(_dets06):
                _xyxy06  = _dets06.xyxy.cpu().numpy()
                _confs06 = _dets06.conf.cpu().numpy().reshape(-1, 1)
                _clss06  = _dets06.cls.cpu().numpy().reshape(-1, 1)
                _darr06  = _np06.concatenate([_xyxy06, _confs06, _clss06], axis=1)
            else:
                _darr06 = _np06.empty((0, 6))
            _tracks06 = _tracker06.update(_darr06, _frame06)
            for _t in _tracks06:
                _cid = int(_t[6]) if len(_t) > 6 else -1
                if _cid < 0 or _cid >= N_CLASSES: continue
                _rows06.append({'frame': _fi06, 'track_id': int(_t[4]),
                                'class_id': _cid, 'conf': float(_t[5])})
        _cap06.release()

    else:
        # ── Ultralytics tracker path (REID_OPTION='A' or 'none') ──────────
        import tempfile as _tmp06
        _tcfg06 = _yaml06.safe_load(_tracker_yaml06.read_text()) if _tracker_yaml06.exists() else {}
        # Inject 'model: auto' if with_reid=True (required by newer Ultralytics)
        if _tcfg06.get('with_reid', False) and 'model' not in _tcfg06:
            _tcfg06['model'] = 'auto'
        _tmp_yaml06 = Path(_tmp06.mktemp(suffix='.yaml'))
        _tmp_yaml06.write_text(_yaml06.dump(_tcfg06))
        _model06 = _YOLO06(str(_weights06))
        for _fi06, _r06 in enumerate(
                _model06.track(str(DEMO_VIDEO_06), tracker=str(_tmp_yaml06),
                               conf=_conf06, iou=0.7, stream=True,
                               verbose=False, persist=False), 1):
            if _r06.boxes is None or _r06.boxes.id is None: continue
            for _box06 in _r06.boxes:
                if _box06.id is None: continue
                _rows06.append({'frame': _fi06,
                                'track_id': int(_box06.id.item()),
                                'class_id': int(_box06.cls.item()),
                                'conf':     float(_box06.conf.item())})
        _tmp_yaml06.unlink(missing_ok=True)

    df_tracks06 = pd.DataFrame(_rows06)
    print(f'  → {len(df_tracks06)} detections, '
          f'{df_tracks06["track_id"].nunique() if not df_tracks06.empty else 0} unique tracks')

    if not df_tracks06.empty:
        # ── Accumulate posteriors frame by frame ──────────────────────────
        _df_u06   = df_tracks06.drop_duplicates(subset=['frame','track_id'])
        _max_f06  = int(_df_u06['frame'].max())
        _counts06 = np.zeros(N_CLASSES, dtype=int)
        _seen06   = set()
        _posts06  = []
        for _f in range(_max_f06 + 1):
            for _, _row in _df_u06[_df_u06.frame == _f].iterrows():
                _tid = int(_row['track_id'])
                if _tid not in _seen06:
                    _counts06[int(_row['class_id'])] += 1
                    _seen06.add(_tid)
            _posts06.append(compute_posterior(_counts06.copy(), PRIOR, LIKELIHOODS))
        _posts06 = np.array(_posts06)
        print(f'  Processed {_max_f06+1} frames, {len(_seen06)} unique tracks counted')

        fig, ax = plt.subplots(figsize=(14, 5))
        for i, mg in enumerate(MINIGAMES):
            ax.plot(_posts06[:, i], label=mg, lw=2)
        ax.set_xlabel('Frame'); ax.set_ylabel('P(minigame)')
        ax.set_title(f'Posterior over Time — Baseline Tracker [{_reid_opt06}] ({DEMO_VIDEO_06.name})',
                     fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
        plot_posterior(_posts06[-1], _counts06,
                       title=f'Final Posterior — Baseline Tracker ({DEMO_VIDEO_06.name})')

    # ── Expose for Section 4 ──────────────────────────────────────────────
    GT_SEQUENCE = DEMO_VIDEO_06.stem


## 4 — Tracker vs Ground Truth: Side-by-Side Posterior Comparison

Loads both the tracker output (Section 3) and the MOT-format ground-truth for the
same video, computes independent cumulative counts and rolling posteriors for each,
then plots them side by side.

**What to look for:**
- Lines that track closely → tracker faithfully counting stats
- Posterior lag or divergence → over/under-counting by the tracker

In [ ]:
from pathlib import Path

# ── Point at the same video used for tracking in Section 3 ─────────────────
# The MOT sequence name should match the video chunk (e.g. "AY_G1_chunk_0003")
# Leave as None to auto-detect from TRACKING_CSV filename.
GT_SEQUENCE = None   # e.g. "AY_G1_chunk_0003"  — set explicitly if needed

MOT_DIR = REPO_ROOT / "data" / "mot_dataset" / "sequences"


def load_gt_counts_per_frame(sequence_name, mot_dir, n_classes):
    """
    Read MOT-format ground-truth labels for a sequence.
    Returns a dict {frame_idx: [class_id, ...]} of new track IDs seen per frame.
    """
    gt_dir = Path(mot_dir) / sequence_name / "gt"
    gt_file = gt_dir / "gt.txt"
    if not gt_file.exists():
        # fallback: look for per-frame label txts
        gt_file = None
        label_dir = Path(mot_dir) / sequence_name / "labels"
        if not label_dir.exists():
            return None, f"GT not found for sequence: {sequence_name}"

    rows = []
    if gt_file:
        for line in gt_file.read_text().splitlines():
            parts = line.strip().split(",")
            if len(parts) < 7: continue
            frame_id, track_id, cls_id = int(parts[0])-1, int(parts[1]), int(parts[7])-1 if len(parts)>7 else 0
            rows.append((frame_id, track_id, cls_id))
    else:
        for txt in sorted(label_dir.glob("*.txt")):
            frame_id = int(txt.stem)
            for line in txt.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    rows.append((frame_id, -1, int(parts[0])))  # no track id in label files

    return rows, None


def accumulate_posteriors(rows, n_classes, prior, likelihoods, max_frame=None):
    """
    Given a list of (frame_idx, track_id, class_id) rows,
    accumulate per-new-track counts and compute posterior at each frame.
    Returns (posteriors array, final counts array).
    """
    if max_frame is None:
        max_frame = max(r[0] for r in rows) if rows else 0
    counts    = np.zeros(n_classes, dtype=int)
    seen      = set()
    posts     = []
    row_map   = {}
    for frame_idx, track_id, cls_id in rows:
        row_map.setdefault(frame_idx, []).append((track_id, cls_id))

    for f in range(max_frame + 1):
        for tid, cid in row_map.get(f, []):
            key = (tid, cid) if tid >= 0 else (f, cid)  # label-only: use frame as pseudo-ID
            if key not in seen:
                counts[cid] += 1
                seen.add(key)
        posts.append(compute_posterior(counts.copy(), prior, likelihoods))
    return np.array(posts), counts.copy()


# ── Auto-detect sequence name from tracking CSV path ───────────────────────
if GT_SEQUENCE is None and "TRACKING_CSV" in dir() and Path(TRACKING_CSV).exists():
    # Try to infer from the CSV parent folder or filename
    GT_SEQUENCE = Path(TRACKING_CSV).stem.replace("tracking_results", "").strip("_") or None

if GT_SEQUENCE is None:
    # Fall back to the first available sequence
    seqs = sorted(MOT_DIR.glob("*/gt/gt.txt")) + sorted(MOT_DIR.glob("*/labels/*.txt"))
    if seqs:
        GT_SEQUENCE = seqs[0].parts[-3]
        print(f"Auto-selected sequence: {GT_SEQUENCE}")

if GT_SEQUENCE:
    gt_rows, err = load_gt_counts_per_frame(GT_SEQUENCE, MOT_DIR, N_CLASSES)
    if err:
        print(f"Ground truth error: {err}")
    else:
        gt_max = max(r[0] for r in gt_rows)

        # Tracker posteriors — use live baseline results from Section 3
        trk_rows = []
        if "df_tracks06" in dir() and df_tracks06 is not None:
            df_t = df_tracks06.drop_duplicates(subset=["frame", "track_id"])
            trk_rows = list(zip(df_t["frame"].astype(int),
                                df_t["track_id"].astype(int),
                                df_t["class_id"].astype(int)))

        common_max = max(gt_max, max((r[0] for r in trk_rows), default=0))

        gt_posts,  gt_counts  = accumulate_posteriors(gt_rows,  N_CLASSES, PRIOR, LIKELIHOODS, common_max)
        trk_posts, trk_counts = accumulate_posteriors(trk_rows, N_CLASSES, PRIOR, LIKELIHOODS, common_max)

        # ── Side-by-side posterior evolution plot ──────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
        colors = plt.cm.Set1.colors
        for i, mg in enumerate(MINIGAMES):
            c = colors[i % len(colors)]
            axes[0].plot(gt_posts[:,  i], label=mg, lw=2, color=c)
            axes[1].plot(trk_posts[:, i], label=mg, lw=2, color=c, ls='--')

        for ax, title in zip(axes, ["Ground Truth", "Tracker Output"]):
            ax.set_xlabel("Frame"); ax.set_ylabel("P(minigame)")
            ax.set_title(f"Posterior over Time — {title}", fontweight="bold")
            ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
        plt.suptitle(f"Sequence: {GT_SEQUENCE}", fontweight="bold", y=1.02)
        plt.tight_layout()
        out = TRACKER_RUNS / f"gt_vs_tracker_posterior_{GT_SEQUENCE}.png" if "TRACKER_RUNS" in dir() \
              else REPO_ROOT / "runs" / f"gt_vs_tracker_posterior_{GT_SEQUENCE}.png"
        Path(out).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
        print(f"Saved: {out}")

        # ── Final bar comparison ────────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
        for ax, posts, counts, title in [
                (axes[0], gt_posts,  gt_counts,  "Ground Truth"),
                (axes[1], trk_posts, trk_counts, "Tracker Output")]:
            bars = ax.bar(MINIGAMES, posts[-1],
                          color=[colors[i % len(colors)] for i in range(len(MINIGAMES))],
                          edgecolor="white")
            for bar, p in zip(bars, posts[-1]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f"{p:.1%}", ha="center", fontsize=9, fontweight="bold")
            ax.set_ylim(0, 1.05); ax.set_ylabel("Probability")
            ax.set_title(f"Final Posterior — {title}  (n={int(counts.sum())} obs)", fontweight="bold")
            ax.grid(axis="y", alpha=0.3)
        plt.suptitle(f"GT vs Tracker: {GT_SEQUENCE}", fontweight="bold", y=1.02)
        plt.tight_layout(); plt.show()

        # ── Stat count comparison table ─────────────────────────────────────
        df_counts = pd.DataFrame({
            "GT counts":      gt_counts,
            "Tracker counts": trk_counts,
            "Difference":     trk_counts - gt_counts,
            "% Error":        np.where(gt_counts > 0,
                                       (trk_counts - gt_counts) / gt_counts * 100, np.nan),
        }, index=CLASSES)
        print("\nStat count comparison:")
        display(df_counts.style.format({"% Error": "{:.1f}%"})
                               .background_gradient(subset=["Difference"], cmap="RdYlGn_r"))
else:
    print("Set GT_SEQUENCE to a sequence name from data/mot_dataset/sequences/")

